#36615 - Final Project Part 3 - Delay Prediction Spark ML
## Team Polyhymnia
### Vraj Patel, Jay Louissant, Elaine Yin

Goal: Use Spark ML with engineered features to predict
whether a flight will be delayed (including cancellations).

We implement several optimizations:
1. **Quantile-based split** (avoids global sort and window functions).
2. **Cache + materialize once** after expensive steps.
3. **Fit preprocessing/scaling on train only** (prevents leakage).
4. **Tune on a sample first**

In [0]:
import pyspark.sql.functions as F
from pyspark.ml.feature import RFormula, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline

from pyspark.ml.classification import LinearSVC

import pandas as pd


#### 1. Load feature-engineered data

In [0]:
df = spark.table("lsd_2026.default.polyhymnia_feature_engineered")

print(f"Rows: {df.count():,}")
df.printSchema()

In [0]:
from pyspark.sql import functions as F

df.agg(
    F.max("FL_DATE").alias("latest_date"),
    F.min("FL_DATE").alias("earliest_date")
).show()

#### 2. Define target label: departure delayed or not

In [0]:
df = df.withColumn(
    "dep_delayed",
    F.when((F.col("DEP_DELAY") > 0) | (F.col("CANCELLED") == 1), F.lit(1)).otherwise(F.lit(0))
)

df.groupBy("dep_delayed").count().show()

#### Logistic Regression

In [0]:
# 2. Train / Test / Validation Split Strategy

# Train: Before 2015
train = df.filter(F.col("FL_DATE") < "2015-01-01").cache()

# Test: 2015 - 2016 (For tuning)
test = df.filter(
    (F.col("FL_DATE") >= "2015-01-01") &
    (F.col("FL_DATE") <  "2017-01-01")
).cache()

# Validation: 2017 onwards
valid = df.filter(F.col("FL_DATE") >= "2017-01-01").cache()

print(f"Train rows: {train.count():,}")
print(f"Test rows:  {test.count():,}")
print(f"Valid rows: {valid.count():,}")


# 3. RFormula and StandardScaler Preprocessing
formula_str = """
dep_delayed ~ 
    OP_CARRIER + 
    CRS_DEP_TIME + CRS_ARR_TIME + CRS_ELAPSED_TIME + DISTANCE +
    day_of_week + hour_of_day +
    crs_dep_time_bucket + arr_time_bucket +
    rate_weather_delay_dep + rate_weather_delay_arr +
    dep_traffic_z_score + carrier_delay_rate_7d + route_delay_rate_7d
"""

# Step 1: RFormula (creates "raw_features")
rf_stage = RFormula(
    formula=formula_str,
    featuresCol="raw_features",
    labelCol="label",
    handleInvalid="skip"
)

rf_model = rf_stage.fit(train)

train_rf = rf_model.transform(train)
test_rf  = rf_model.transform(test)
valid_rf = rf_model.transform(valid)

# Step 2: StandardScaler (creates final "features" for LR)
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True)
scaler_model = scaler.fit(train_rf)

train_fe = scaler_model.transform(train_rf).select("label", "features")
test_fe  = scaler_model.transform(test_rf).select("label", "features")
valid_fe = scaler_model.transform(valid_rf).select("label", "features")

print("Scaling and Feature matrices read.")


# 4. Logistic Regression Tuning (Grid Search on Test Set)
train_sample_fe = train_fe.sample(withReplacement=False, fraction=0.10, seed=42).cache()

lr = LogisticRegression(labelCol="label", featuresCol="features", maxIter=15)

reg_params = [0.01, 0.1]
elastic_net_params = [0.0, 0.5, 1.0]

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

best_f1 = -1.0
best_params = {}

print("Starting Parameter Tuning on 10% Train Sample...")
for reg in reg_params:
    for en in elastic_net_params:
        lr_curr = lr.copy({lr.regParam: reg, lr.elasticNetParam: en})
        
        model_temp = lr_curr.fit(train_sample_fe)
        preds = model_temp.transform(test_fe)
        f1_score = evaluator.evaluate(preds)
        
        print(f"LR (reg={reg}, elasticNet={en}) | Test F1: {f1_score:.4f}")
        
        if f1_score > best_f1:
            best_f1 = f1_score
            best_params = {'regParam': reg, 'elasticNetParam': en}

print(f"\nBest Parameters Found: {best_params}. Now fitting on FULL Training data...")

train_sample_fe.unpersist()

best_lr = LogisticRegression(labelCol="label", featuresCol="features", maxIter=15, 
                             regParam=best_params['regParam'], 
                             elasticNetParam=best_params['elasticNetParam'])

best_model = best_lr.fit(train_fe)

print("Full model training completed successfully!")




# 5. Evaluate Best Model on Validation Set 
valid_preds = best_model.transform(valid_fe)

# Confusion Matrix
cm = (valid_preds
      .groupBy("label", "prediction")
      .count()
      .toPandas())

print("Confusion Matrix (Validation):")
print(cm)

TP = cm[(cm["label"] == 1.0) & (cm["prediction"] == 1.0)]["count"].sum()
TN = cm[(cm["label"] == 0.0) & (cm["prediction"] == 0.0)]["count"].sum()
FP = cm[(cm["label"] == 0.0) & (cm["prediction"] == 1.0)]["count"].sum()
FN = cm[(cm["label"] == 1.0) & (cm["prediction"] == 0.0)]["count"].sum()

TP, TN, FP, FN = map(float, [TP, TN, FP, FN])

accuracy = (TP + TN) / (TP + TN + FP + FN)
tpr = TP / (TP + FN) if (TP + FN) > 0 else 0        
fpr = FP / (FP + TN) if (FP + TN) > 0 else 0
specificity = TN / (TN + FP) if (TN + FP) > 0 else 0

valid_f1 = evaluator.evaluate(valid_preds)

print("\nValidation Metrics (Logistic Regression):")
print(f"Accuracy:           {accuracy:.4f}")
print(f"F1 Score:           {valid_f1:.4f}")
print(f"TPR/Sensitivity:    {tpr:.4f}")
print(f"False Positive Rate:{fpr:.4f}")
print(f"Specificity:        {specificity:.4f}")


# 6. Metrics for Baseline Model

baseline = valid_fe.withColumn("prediction", F.lit(0.0))

baseline_cm = (baseline
               .groupBy("label", "prediction")
               .count()
               .toPandas())

print("Baseline Confusion Matrix:")
print(baseline_cm)

B_TP = baseline_cm[(baseline_cm["label"] == 1.0) & (baseline_cm["prediction"] == 1.0)]["count"].sum()
B_TN = baseline_cm[(baseline_cm["label"] == 0.0) & (baseline_cm["prediction"] == 0.0)]["count"].sum()
B_FP = baseline_cm[(baseline_cm["label"] == 0.0) & (baseline_cm["prediction"] == 1.0)]["count"].sum()
B_FN = baseline_cm[(baseline_cm["label"] == 1.0) & (baseline_cm["prediction"] == 0.0)]["count"].sum()

B_acc = (B_TP + B_TN) / (B_TP + B_TN + B_FP + B_FN)
B_tpr = B_TP / (B_TP + B_FN) if (B_TP + B_FN) > 0 else 0
B_fpr = B_FP / (B_FP + B_TN) if (B_FP + B_TN) > 0 else 0
B_spec = B_TN / (B_TN + B_FP) if (B_TN + B_FP) > 0 else 0

print("\nBaseline Metrics:")
print(f"Accuracy:           {B_acc:.4f}")
print(f"TPR/Sensitivity:    {B_tpr:.4f}")
print(f"True Negative Rate: {B_spec:.4f}")


# 7. Comparison Summary
print("\nValidation Performance Comparison (LR vs Baseline):")
print(f"LR Accuracy vs Baseline: {accuracy:.4f} vs {B_acc:.4f}")
print(f"LR TPR vs Baseline:      {tpr:.4f} vs {B_tpr:.4f}")
print(f"LR FPR vs Baseline:      {fpr:.4f} vs {B_fpr:.4f}")
print(f"LR Specificity vs Base:  {specificity:.4f} vs {B_spec:.4f}")

#### SVM

In [0]:
import pyspark.sql.functions as F
from pyspark.ml.feature import RFormula, StandardScaler
from pyspark.ml.classification import LinearSVC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
import pandas as pd

# 1. Load feature-engineered data

df = spark.table("lsd_2026.default.polyhymnia_feature_engineered")

df = df.withColumn(
    "dep_delayed",
    F.when((F.col("DEP_DELAY") > 0) | (F.col("CANCELLED") == 1), F.lit(1)).otherwise(F.lit(0))
)


# 2. Train / Test / Validation Split Strategy (Date-based matching Person 2)

# Train: Before 2015
train = df.filter(F.col("FL_DATE") < "2015-01-01").cache()

# Test: 2015 - 2016 (For tuning)
test = df.filter(
    (F.col("FL_DATE") >= "2015-01-01") &
    (F.col("FL_DATE") <  "2017-01-01")
).cache()

# Validation: 2017 onwards
valid = df.filter(F.col("FL_DATE") >= "2017-01-01").cache()

print(f"Train rows: {train.count():,}")
print(f"Test rows:  {test.count():,}")
print(f"Valid rows: {valid.count():,}")

 
# 3. RFormula and StandardScaler Preprocessing
formula_str = """
dep_delayed ~ 
    OP_CARRIER + 
    CRS_DEP_TIME + CRS_ARR_TIME + CRS_ELAPSED_TIME + DISTANCE +
    day_of_week + hour_of_day +
    crs_dep_time_bucket + arr_time_bucket +
    rate_weather_delay_dep + rate_weather_delay_arr +
    dep_traffic_z_score + carrier_delay_rate_7d + route_delay_rate_7d
"""

# Step 1: RFormula 
rf_stage = RFormula(
    formula=formula_str,
    featuresCol="raw_features",
    labelCol="label",
    handleInvalid="skip"
)

rf_model = rf_stage.fit(train)

train_rf = rf_model.transform(train)
test_rf  = rf_model.transform(test)
valid_rf = rf_model.transform(valid)

# Step 2: StandardScaler
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True)
scaler_model = scaler.fit(train_rf)

train_fe = scaler_model.transform(train_rf).select("label", "features")
test_fe  = scaler_model.transform(test_rf).select("label", "features").cache()
valid_fe = scaler_model.transform(valid_rf).select("label", "features").cache()


# 4. LinearSVC Tuning 

# Sample 10% of training data for tuning
train_sample_fe = train_fe.sample(withReplacement=False, fraction=0.10, seed=42).cache()

# Define Linear Support Vector Machine
lsvc = LinearSVC(labelCol="label", featuresCol="features")

# Hyperparameters: regParam (Regularization) and maxIter
reg_params = [0.01, 0.1]
iter_params = [10, 20]

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

best_f1 = -1.0
best_params = {}

print("Starting SVM Parameter Tuning on 10% Train Sample...")
for reg in reg_params:
    for max_i in iter_params:
        lsvc_curr = lsvc.copy({lsvc.regParam: reg, lsvc.maxIter: max_i})
        
        # Fit on sample
        model_temp = lsvc_curr.fit(train_sample_fe)
        
        # Evaluate on Test set
        preds = model_temp.transform(test_fe)
        f1_score = evaluator.evaluate(preds)
        
        print(f"LinearSVC (regParam={reg}, maxIter={max_i}) | Test F1: {f1_score:.4f}")
        
        if f1_score > best_f1:
            best_f1 = f1_score
            best_params = {'regParam': reg, 'maxIter': max_i}

print(f"\nBest Parameters Found: {best_params}. Now fitting on FULL Training data...")

# Unpersist sample to free memory
train_sample_fe.unpersist()

# Fit the optimal model on FULL training set
best_lsvc = LinearSVC(labelCol="label", featuresCol="features", 
                      regParam=best_params['regParam'], maxIter=best_params['maxIter'])

best_model = best_lsvc.fit(train_fe)
print("Full SVM model training completed successfully!")




# 5. Evaluate Best Model on Validation Set 

valid_preds = best_model.transform(valid_fe)

# Confusion Matrix via Pandas
cm = (valid_preds
      .groupBy("label", "prediction")
      .count()
      .toPandas())

print("Confusion Matrix (Validation):")
print(cm)

TP = cm[(cm["label"] == 1.0) & (cm["prediction"] == 1.0)]["count"].sum()
TN = cm[(cm["label"] == 0.0) & (cm["prediction"] == 0.0)]["count"].sum()
FP = cm[(cm["label"] == 0.0) & (cm["prediction"] == 1.0)]["count"].sum()
FN = cm[(cm["label"] == 1.0) & (cm["prediction"] == 0.0)]["count"].sum()

TP, TN, FP, FN = map(float, [TP, TN, FP, FN])

accuracy = (TP + TN) / (TP + TN + FP + FN)
tpr = TP / (TP + FN) if (TP + FN) > 0 else 0        
fpr = FP / (FP + TN) if (FP + TN) > 0 else 0
specificity = TN / (TN + FP) if (TN + FP) > 0 else 0

valid_f1 = evaluator.evaluate(valid_preds)

print("\nValidation Metrics (LinearSVC):")
print(f"Accuracy:           {accuracy:.4f}")
print(f"F1 Score:           {valid_f1:.4f}")
print(f"TPR/Sensitivity:    {tpr:.4f}")
print(f"False Positive Rate:{fpr:.4f}")
print(f"Specificity:        {specificity:.4f}")


# 6. Metrics for Baseline Model 

baseline = valid_fe.withColumn("prediction", F.lit(0.0))

baseline_cm = (baseline
               .groupBy("label", "prediction")
               .count()
               .toPandas())

print("Baseline Confusion Matrix:")
print(baseline_cm)

B_TP = baseline_cm[(baseline_cm["label"] == 1.0) & (baseline_cm["prediction"] == 1.0)]["count"].sum()
B_TN = baseline_cm[(baseline_cm["label"] == 0.0) & (baseline_cm["prediction"] == 0.0)]["count"].sum()
B_FP = baseline_cm[(baseline_cm["label"] == 0.0) & (baseline_cm["prediction"] == 1.0)]["count"].sum()
B_FN = baseline_cm[(baseline_cm["label"] == 1.0) & (baseline_cm["prediction"] == 0.0)]["count"].sum()

B_acc = (B_TP + B_TN) / (B_TP + B_TN + B_FP + B_FN)
B_tpr = B_TP / (B_TP + B_FN) if (B_TP + B_FN) > 0 else 0
B_fpr = B_FP / (B_FP + B_TN) if (B_FP + B_TN) > 0 else 0
B_spec = B_TN / (B_TN + B_FP) if (B_TN + B_FP) > 0 else 0

print("\nBaseline Metrics:")
print(f"Accuracy:           {B_acc:.4f}")
print(f"TPR/Sensitivity:    {B_tpr:.4f}")
print(f"True Negative Rate: {B_spec:.4f}")


# 7. Comparison Summary

print("\n=== Validation Performance Comparison (LinearSVC vs Baseline) ===")
print(f"SVM Accuracy vs Baseline: {accuracy:.4f} vs {B_acc:.4f}")
print(f"SVM TPR vs Baseline:      {tpr:.4f} vs {B_tpr:.4f}")
print(f"SVM FPR vs Baseline:      {fpr:.4f} vs {B_fpr:.4f}")
print(f"SVM Specificity vs Base:  {specificity:.4f} vs {B_spec:.4f}")